# Stage 1 — Informed minority replication (Oct–Dec 2024)

Orchestration and prerequisites: **`docs/SETUP.md`**.

Pipeline: Gamma → R2 metadata → `order_filled` on R2 (historical Goldsky ingest; Goldsky is no longer used) → DuckDB analysis (PnL, event-level sign randomization, classification, OIB regression).

**Notebook layout:** run the **setup** cell first, then either **Pre-R2** (metadata + ingest checks) or **Post-R2** (DuckDB analysis only — assumes `order_filled/` is already on R2).

**Train / test (minority validation):** set `ANALYSIS_BLOCK_TS_MIN` / `MAX` (or `ANALYSIS_USE_LAST_N_DAYS`), `MINORITY_LABEL_SPLIT=train_3w_test_1w`, then run `pnl` → `sign_randomization` → `classification` (labels from **train** weeks). For out-of-time OIB, set `OIB_EVAL_WEEK=test` and run `oib_regression`.

In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd()
if (ROOT / "pyproject.toml").exists():
    pass
elif (ROOT.parent / "pyproject.toml").exists():
    os.chdir(ROOT.parent)
load_dotenv()

from src.lib.duck import get_duck, r2_uri

print("CWD", Path.cwd())
print("R2_BUCKET", os.environ.get("R2_BUCKET", "(unset)"))

CWD /Users/qtpie/repos/polymarket
R2_BUCKET polymarket


### Pre-R2 — metadata and ingest checks

Writes **metadata** to R2 and runs **`verify_ingest`** on fills already on R2. Set `run_pre_r2(run=True)` when you want this path.

In [ ]:
def run_pre_r2(run: bool = False):
    """Gamma → R2 metadata, verify Parquet on R2. Does not run DuckDB analysis."""
    if not run:
        print("Skipping pre-R2 (run=False). Set run=True after .env is ready.")
        return
    from src.data import load_markets, verify_ingest

    load_markets.main()
    verify_ingest.main()


run_pre_r2(run=False)


### Post-R2 — DuckDB analysis only

Assumes **`order_filled/**`** (and metadata under **`metadata/`**) already exist on R2. Run the **setup** cell above first. Set `run_post_r2_analysis(run=True)` to execute: `partition` → `pnl` → `sign_randomization` → `classification` → `oib_regression`.

In [ ]:
def run_post_r2_analysis(run: bool = False):
    """Partition + PnL → sign randomization → classification → OIB (reads/writes derived + labels on R2)."""
    if not run:
        print("Skipping post-R2 analysis (run=False). Set run=True when order_filled is on R2.")
        return
    from src.analysis import partition, pnl, sign_randomization, classification, oib_regression

    partition.main()
    pnl.main()
    sign_randomization.main()
    classification.main()
    oib_regression.main()


run_post_r2_analysis(run=False)

In [ ]:
# PnL histogram by trader group (requires labels on R2)
con = get_duck()
try:
    acct = con.execute(f"SELECT * FROM read_parquet('{r2_uri('derived', 'pnl_account.parquet')}')").df()
    lbl = con.execute(f"SELECT * FROM read_parquet('{r2_uri('labels', 'account_classification.parquet')}')").df()
    m = acct.merge(lbl[["user_id", "trader_group"]], on="user_id", how="left")
    fig, ax = plt.subplots(figsize=(8, 4))
    for g in m["trader_group"].dropna().unique():
        sub = m.loc[m["trader_group"] == g, "total_pnl"].clip(-5_000, 5_000)
        if len(sub) > 0:
            ax.hist(sub, bins=40, alpha=0.35, label=str(g), density=True)
    ax.set_xlabel("total_pnl (clipped ±5000)")
    ax.legend(fontsize=7)
    ax.set_title("Realized PnL by trader_group")
    plt.show()
except Exception as e:
    print("Skip plot:", e)

In [3]:
# Skilled share banner + go / no-go (reads classification parquet)
try:
    con = get_duck()
    lbl = con.execute(f"SELECT * FROM read_parquet('{r2_uri('labels', 'account_classification.parquet')}')").df()
    classified = lbl[lbl["trader_group"] != "Excluded"]
    share = (lbl["trader_group"] == "Skilled_Winner").mean()
    share_c = (lbl["trader_group"] == "Skilled_Winner").sum() / max(len(classified), 1)
    print(f"Skilled share (all rows): {share:.2%}")
    print(f"Skilled share / non-excluded: {share_c:.2%}")
    band_ok = 0.02 <= share_c <= 0.06
    print("BAND_OK (2–6% on non-excluded):", band_ok)
except Exception as e:
    print("Classification not available:", e)

Skilled share (all rows): 0.92%
Skilled share / non-excluded: 2.48%
BAND_OK (2–6% on non-excluded): True


In [4]:
# Run Report: concatenate recent JSONL debug logs
from glob import glob

lines = []
for path in sorted(glob("logs/*.jsonl"))[-15:]:
    lines.append(f"## {path}\n")
    with open(path, encoding="utf-8") as f:
        tail = f.readlines()[-40:]
    lines.extend(tail)
    lines.append("\n")
report = "".join(lines)
print(report[:8000] if len(report) > 8000 else report)

## logs/classification_20260515T033834Z.jsonl
{"step": "classification", "phase": "ingest", "metric": "n_accounts", "value": 259415, "notes": null}
{"step": "classification", "phase": "ingest", "metric": "n_sign_randomization_rows", "value": 153261, "notes": null}
{"step": "classification", "phase": "diagnostic", "metric": "saved_parquet", "value": "logs/classification/group_counts.parquet", "notes": null, "table": "group_counts"}
{"step": "classification", "phase": "group", "metric": "Excluded", "value": 163023, "notes": null}
{"step": "classification", "phase": "group", "metric": "Unlucky_Loser", "value": 50198, "notes": null}
{"step": "classification", "phase": "group", "metric": "Unskilled_Loser", "value": 23743, "notes": null}
{"step": "classification", "phase": "group", "metric": "Lucky_Winner", "value": 20061, "notes": null}
{"step": "classification", "phase": "group", "metric": "Skilled_Winner", "value": 2390, "notes": null}
{"step": "classification", "phase": "metric", "metric

In [5]:
oib_df = pd.read_parquet('logs/oib_regression/regression_full_FE.parquet')
print(oib_df.head())

                   term     param    stderr     tstat    pvalue
0           OIB_skilled -0.001537  0.002026 -0.758381  0.448226
1             OIB_lucky -0.001337  0.001206 -1.108713  0.267559
2           OIB_unlucky -0.004513  0.001220 -3.697632  0.000218
3         OIB_unskilled -0.006212  0.001465 -4.240252  0.000022
4  volume_share_skilled -0.002154  0.001372 -1.570494  0.116306


In [2]:
oib_df = pd.read_parquet('logs/oib_regression/regression_full_FE.parquet')
print(oib_df.head())

                   term     param    stderr     tstat    pvalue
0           OIB_skilled -0.000001  0.002338 -0.000579  0.999538
1             OIB_lucky -0.001529  0.001251 -1.221883  0.221757
2           OIB_unlucky -0.006073  0.001263 -4.807988  0.000002
3         OIB_unskilled -0.002904  0.002082 -1.394954  0.163035
4  volume_share_skilled -0.004170  0.001559 -2.675275  0.007469
